In [ ]:
import json
import time
import ollama
import chromadb
from chromadb.config import Settings
from ollama._types import EmbeddingsResponse
from collections import Counter
from typing import List

# OneOrMany = Union[T, List[T]]
# --- Config ---
CHUNKS_PATH = "../data/rag/to_ingest/chunks_all.json"
CHROMA_DB_PATH = "../chroma_db"  # persisted locally
COLLECTION_NAME = "socratic_tutor_collection"
EMBED_MODEL = "qwen3-embedding:0.6b"  # ollama pull qwen3-embedding:0.6b
BATCH_SIZE = 50  # chunks per embedding batch

---
## 2. Load & Clean Chunks

In [2]:
with open(CHUNKS_PATH, encoding="utf-8") as f:
    raw_chunks = json.load(f)

print(f"Raw chunks loaded: {len(raw_chunks)}")

# Filter chunks too short to be meaningful (noise)
MIN_LENGTH = 100
chunks = [c for c in raw_chunks if len(c["content"]) >= MIN_LENGTH]

print(f"After filtering  : {len(chunks)}")
print(f"Removed          : {len(raw_chunks) - len(chunks)}")


dist = Counter(c["id"] for c in chunks)
for ch, count in sorted(dist.items()):
    has_code = sum(1 for c in chunks if c["id"] == ch and c["has_code"])
    print(f"  {ch:<20} total={count:<5} with_code={has_code}")

Raw chunks loaded: 2720
After filtering  : 2631
Removed          : 89
  chunk_00000          total=1     with_code=0
  chunk_00001          total=1     with_code=1
  chunk_00002          total=1     with_code=0
  chunk_00003          total=1     with_code=1
  chunk_00004          total=1     with_code=0
  chunk_00005          total=1     with_code=1
  chunk_00006          total=1     with_code=0
  chunk_00007          total=1     with_code=1
  chunk_00009          total=1     with_code=1
  chunk_00010          total=1     with_code=0
  chunk_00011          total=1     with_code=1
  chunk_00012          total=1     with_code=0
  chunk_00013          total=1     with_code=1
  chunk_00015          total=1     with_code=1
  chunk_00016          total=1     with_code=0
  chunk_00017          total=1     with_code=1
  chunk_00018          total=1     with_code=0
  chunk_00019          total=1     with_code=1
  chunk_00020          total=1     with_code=1
  chunk_00022          total=1     wi

## 3. Embed & Ingest into ChromaDB


In [19]:
def get_embedding(text: str, is_query: bool = False) -> List[float]:
    """
    Get embedding using Qwen3-Embedding instruction format.
    
    Queries receive a task instruction prefix for better retrieval.
    Documents are embedded as-is per Qwen3 spec.
    """
    QUERY_INSTRUCTION = (
        "Instruct: Given a student question about C programming, "
        "retrieve the most relevant educational passages that answer the question.\n"
        "Query: "
    )

    pmt = f"{QUERY_INSTRUCTION}{text}" if is_query else text
    response = ollama.embeddings(model=EMBED_MODEL, prompt=pmt)
    return response["embedding"]


def embed_batch(texts: List[str], is_query: bool = False) -> List[List[float]]:
    return [get_embedding(t, is_query=is_query) for t in texts]


In [17]:
# Initialize ChromaDB (persistent local)
client = chromadb.PersistentClient(
    path=CHROMA_DB_PATH, settings=Settings(anonymized_telemetry=False)
)
collection = client.get_collection(name=COLLECTION_NAME)

In [ ]:
collection

Collection(name=socratic_tutor_collection)

In [ ]:
# Initialize ChromaDB (persistent local)
client = chromadb.PersistentClient(
    path=CHROMA_DB_PATH, settings=Settings(anonymized_telemetry=False)
)

# Drop and recreate collection for clean ingestion
# Comment out the delete line if you want to resume a partial ingestion
try:
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing collection '{COLLECTION_NAME}'")
except:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},  # cosine distance for semantic search
)
print(f"Collection '{COLLECTION_NAME}' created.")

Deleted existing collection 'socratic_tutor_collection'
Collection 'socratic_tutor_collection' created.


In [5]:
import time as _time

# ── Pre-flight token guard ───────────────────────────────────────────────────────────────────────
# Use the same tokenizer as the chunking notebook to pre-screen chunks.
# EMBED_MAX_TOKENS = 8138 (8192 hard limit - 4 prefix tokens - 50 safety margin)
_EMBED_PREFIX = "search_document: "
_EMBED_MAX = 8138

try:
    import tiktoken as _tiktoken
    _tok = _tiktoken.get_encoding("cl100k_base")
    def _embed_token_count(text: str) -> int:
        return len(_tok.encode(_EMBED_PREFIX + text))
except ImportError:
    # Fallback: conservative character-based estimate (4 chars ≈ 1 token)
    def _embed_token_count(text: str) -> int:
        return len(_EMBED_PREFIX + text) // 4

pre_skipped = []
safe_chunks = []
for c in chunks:
    et = _embed_token_count(c["content"])
    if et > _EMBED_MAX:
        pre_skipped.append({"id": c["id"], "embed_tokens": et, "chars": len(c["content"])})
    else:
        safe_chunks.append(c)

if pre_skipped:
    print(f"Pre-flight: skipping {len(pre_skipped)} chunks that exceed embed limit ({_EMBED_MAX}t):")
    for s in pre_skipped:
        print(f"  {s['id']} | embed={s['embed_tokens']}t | chars={s['chars']}")
else:
    print(f"Pre-flight: all {len(chunks)} chunks within embed limit \u2714")

chunks_to_ingest = safe_chunks
print(f"Chunks to ingest: {len(chunks_to_ingest)}")

Pre-flight: all 2631 chunks within embed limit ✔
Chunks to ingest: 2631


In [6]:
import time as _time

MAX_RETRIES = 2
RETRY_DELAY = 2.0  # seconds, doubled on each retry

def embed_with_retry(text: str, is_query: bool = False) -> list:
    """Embed a single text with exponential backoff retry for transient errors."""
    last_err = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            return get_embedding(text, is_query=is_query)
        except Exception as e:
            err_str = str(e).lower()
            # Don't retry oversized content errors — they will never succeed
            if "input length exceeds" in err_str or "context length" in err_str:
                raise
            last_err = e
            if attempt < MAX_RETRIES:
                wait = RETRY_DELAY * (2 ** attempt)
                print(f"  Transient error (attempt {attempt+1}/{MAX_RETRIES+1}), retrying in {wait:.1f}s: {e}")
                _time.sleep(wait)
    raise last_err

total = len(chunks_to_ingest)
ingested = 0
skipped = []
errors = []

start = _time.time()

for idx, c in enumerate(chunks_to_ingest):
    try:
        embedding = embed_with_retry(c["content"], is_query=False)
        metadata = {
            "id": c["id"],
            "heading": c.get("heading", ""),
            "has_code": str(c.get("has_code", False)),
        }
        collection.add(
            ids=[c["id"]],
            embeddings=[embedding],
            documents=[c["content"]],
            metadatas=[metadata],
        )
        ingested += 1
        if ingested % BATCH_SIZE == 0 or ingested == total:
            elapsed = _time.time() - start
            rate = ingested / elapsed if elapsed > 0 else 0
            eta = (total - ingested) / rate if rate > 0 else 0
            print(f"[{ingested:>4}/{total}] {rate:.1f} chunks/s | ETA: {eta:.0f}s")

    except Exception as ex:
        err_str = str(ex)
        skipped.append({"id": c["id"], "error": err_str, "chars": len(c["content"])})
        errors.append({"id": c["id"], "error": err_str})
        print(f"  SKIP {c['id']}: {err_str[:120]}")

print(f"\nIngestion complete.")
print(f"  Ingested : {ingested}/{total}")
print(f"  Skipped  : {len(skipped)}")
if skipped:
    for s in skipped:
        print(f"    {s['id']} | chars={s['chars']} | {s['error'][:80]}")
print(f"  Total time: {_time.time() - start:.1f}s")
print(f"  Collection count: {collection.count()}")

[  50/2631] 4.7 chunks/s | ETA: 548s
[ 100/2631] 5.1 chunks/s | ETA: 493s
[ 150/2631] 4.8 chunks/s | ETA: 513s
[ 200/2631] 4.6 chunks/s | ETA: 528s
[ 250/2631] 4.7 chunks/s | ETA: 509s
[ 300/2631] 4.7 chunks/s | ETA: 491s
[ 350/2631] 4.5 chunks/s | ETA: 503s
[ 400/2631] 3.7 chunks/s | ETA: 597s
[ 450/2631] 3.5 chunks/s | ETA: 624s
[ 500/2631] 3.4 chunks/s | ETA: 628s
[ 550/2631] 3.3 chunks/s | ETA: 624s
[ 600/2631] 3.2 chunks/s | ETA: 625s
[ 650/2631] 3.1 chunks/s | ETA: 638s
[ 700/2631] 3.1 chunks/s | ETA: 633s
[ 750/2631] 3.1 chunks/s | ETA: 612s
[ 800/2631] 3.1 chunks/s | ETA: 594s
[ 850/2631] 3.0 chunks/s | ETA: 594s
[ 900/2631] 3.0 chunks/s | ETA: 579s
  SKIP chunk_01074: the input length exceeds the context length (status code: 500)
[ 950/2631] 2.9 chunks/s | ETA: 587s
  SKIP chunk_01140: the input length exceeds the context length (status code: 500)
  SKIP chunk_01165: the input length exceeds the context length (status code: 500)
[1000/2631] 2.6 chunks/s | ETA: 636s
  SKIP chun

## 4. Retrieval Validation

In [20]:
def query_collection(query: str, n_results: int = 5, only_code: bool = False) -> None:
    """Query ChromaDB and print results with similarity scores."""
    where = {"has_code": "True"} if only_code else None

    query_emb = get_embedding(query, is_query=True)

    results = collection.query(
        query_embeddings=[query_emb],
        n_results=n_results,
        where=where,
        include=["documents", "metadatas", "distances"],
    )

    print(f"\nQuery: '{query}'")
    print(f"{'─' * 70}")

    for i, (doc, meta, dist) in enumerate(
        zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
    ):
        similarity = 1 - dist  # ChromaDB cosine returns distance, not similarity
        has_code = meta["has_code"] == "True"
        print(
            f"[{i + 1}] sim={similarity:.4f} | {meta['id']} | heading: '{meta['heading']}'"
        )
        print(f"     has_code={has_code}")
        print(f"     {doc}...")
        print()

In [8]:
# --- Validation queries ---
# These cover different concept types: memory, syntax, data structures, control flow
validation_queries = [
    "How do pointers work in C?",
    "What is the difference between malloc and calloc?",  
    "How do arrays relate to pointers?",  
    "What is a struct in C?", 
    "How does a for loop work in C?", 
    "What are function pointers?",
    "How do you handle strings in C?", 
    "What is undefined behavior in C?", 
]

for q in validation_queries:
    query_collection(q, n_results=3)


Query: 'How do pointers work in C?'
──────────────────────────────────────────────────────────────────────
[1] sim=0.7397 | chunk_02330 | heading: 'Here are some key points about pointers in C:'
     has_code=True
     ## Here are some key points about pointers in C:

*   **Pointer to Pointer:** Pointers in C can themselves be pointed to by other pointers, creating a chain of references. These are known as "pointer to pointer" or double pointers. For example: In C programming, there are several arithmetic operations you can perform on pointers, but it's essential to be cautious and understand memory layout to avoid accessing invalid memory locations. Here are the main operations:

```c
int x = 10;
int \*ptr1 = &x;
int \*\*ptr2 = &ptr1;

Example:

#include <stdio.h>

int main() {
    int x = 10;
    int \*ptr1 = &x;
    int \*\*ptr2 = &ptr1;

    printf("Value of x: %d\n", x);
    printf("Value pointed by ptr1: %d\n", \*ptr1);
    printf("Value pointed by ptr2: %d\n", \*\*ptr2);

    r

In [9]:
query_collection("C string manipulation char array null terminator", n_results=3)
query_collection("string functions strlen strcpy C", n_results=3)


Query: 'C string manipulation char array null terminator'
──────────────────────────────────────────────────────────────────────
[1] sim=0.7046 | chunk_01644 | heading: 'Output'
     has_code=False
     ## Output

It will produce the following output − The "%s" specifier tells the function to iterate through the array, until it encounters the null terminator (\\0) and printing each character. This effectively prints the entire string represented by the character array without having to use a loop. char greeting\[\] \= {'H', 'e', 'l', 'l', 'o', '\\0'}; printf("Greeting message: %s\\n", greeting ); It will produce the following output − char greeting\[3\] \= {'H', 'e', 'l', 'l', 'o', '\\0'}; printf("%s", greeting); Instead of constructing a char array of individual char values in single quotation marks, and using "\\0" as the last element, C lets you construct a string by enclosing the characters within double quotation marks. This method of initializing a string is more convenient, as 

In [21]:
# Code-specific retrieval — only return chunks that have code examples
# query_collection("pointer arithmetic example", n_results=3, only_code=True)
# query_collection("malloc free memory allocation example", n_results=3, only_code=True)
query_collection("que son los bucles", n_results=1, only_code=True)


Query: 'que son los bucles'
──────────────────────────────────────────────────────────────────────
[1] sim=0.5412 | chunk_02632 | heading: 'for loop'
     has_code=True
     ## for loop

... <statement 1> for(<expression1>; <expression2>; <expression3>) { In the _for_ loop construct, after executing <statement 1>, the C program will jump to execute <expression1>. This expression is executed only once, just before entering the loop for the first time. Thereafter, <expression2> is evaluated. If it evaluates to true, loop is entered and <statement 2> is executed. This could be a set of statements enclosed within the curly braces. After execution of <statement 2>, program will jump to <expression3>, execute it and then return back to <expression2>. If <expression2> evaluates to true again, the program will re-enter the loop and re-execute <statement 2> After execution of <statement 2> again the program will jump to <expression3> & then <expression2>. This looping process of _expression2 -

In [13]:
query_collection("que son los arreglos", n_results=3, only_code=True)


Query: 'que son los arreglos'
──────────────────────────────────────────────────────────────────────
[1] sim=0.5476 | chunk_02784 | heading: 'Pointers in C - Part 4 of 9'
     has_code=True
     ## Pointers in C - Part 4 of 9

Let us consider the following example: A schematic representation of the array x is shown in the following figure:

```c
{1, 2, 3},
{4, 5, 6},
{7, 8, 9}
      };
```...

[2] sim=0.5060 | chunk_01614 | heading: 'Output'
     has_code=True
     ## Output

Two-dimensional array: 92   19   79   23 56   21   44   98 A jagged array is a collection of two or more arrays of similar data types of variable length. In C, the concept of jagged array is implemented with the help of **pointers of arrays**.

```c
8   22   89   54
```...

[3] sim=0.4991 | chunk_02757 | heading: 'Exercise'
     has_code=True
     ## Exercise

Identify the array defined in each of the following statements. Indicate what values are assigned to the individual array elements.

```c
{5,4,3,2},
{9,8,7

In [14]:

query_collection("que son las funciones", n_results=3, only_code=True)


Query: 'que son las funciones'
──────────────────────────────────────────────────────────────────────
[1] sim=0.5984 | chunk_01803 | heading: 'Example: Array of Funciton pointers'
     has_code=True
     ## Example: Array of Funciton pointers

} //second function void sub()  { } //third function void mult() {

```c
printf("Subtract function\\n");
```...

[2] sim=0.5982 | chunk_01801 | heading: 'Example: Array of Funciton pointers'
     has_code=True
     ## Example: Array of Funciton pointers

#include <stdio.h> //first function void add()  { } //second function void sub()  {

```c
printf("Add function\\n");
```...

[3] sim=0.5905 | chunk_02395 | heading: '2.  User-Defined Functions:'
     has_code=True
     ## 2.  User-Defined Functions:

##### Declaration of multiple function and Calling these from one function.

```c
#include

// Function prototype can acept only type as well as variable declaration with in parameter.

void add(int , int ); // only type
void sub(int x, int y); // t